# 02 — Saída estruturada com Pydantic

**Trilha:** [01 chamada e JSON](01_chamada_e_json.ipynb) → **02 saída estruturada** → [03 enums e guardrails](03_enums_e_guardrails.ipynb)

**De onde viemos:** no [01](01_chamada_e_json.ipynb), o JSON pedido no prompt veio com a quantidade como texto (`"250 grams"`),
e nada garantia o formato. Aqui a lista de compras ganha um **contrato**: uma classe Pydantic que diz quais campos existem e de
que tipo — e que confere a resposta.

## 1. O contrato

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # saida.py e logger_general.py ficam em AI_use_cases/
from saida import mostrar, mostrar_prompt, tabela

In [2]:
import json
from pathlib import Path

from pydantic import BaseModel, ConfigDict, Field, ValidationError


class Ingrediente(BaseModel):
    model_config = ConfigDict(extra="forbid")
    nome: str
    quantidade: float = Field(gt=0, description="Quantidade numérica, sem unidade")
    unidade: str = Field(description="Unidade métrica: g, kg, ml, l ou un")


class ListaDeCompras(BaseModel):
    model_config = ConfigDict(extra="forbid")
    prato: str
    porcoes: int = Field(gt=0)
    ingredientes: list[Ingrediente]

Duas mudanças em relação ao 01: a quantidade é **número** e a unidade é um campo **separado**. O Pydantic gera, a partir da
classe, o JSON Schema que descreve esse contrato — é isso que vai para o modelo:

In [3]:
schema = ListaDeCompras.model_json_schema()
mostrar(f"campos de ListaDeCompras: {list(schema['properties'])}")
mostrar(f"campos de Ingrediente: {list(schema['$defs']['Ingrediente']['properties'])}")
mostrar(f"quantidade: {schema['$defs']['Ingrediente']['properties']['quantidade']}")

campos de ListaDeCompras: ['prato', 'porcoes', 'ingredientes']
campos de Ingrediente: ['nome', 'quantidade', 'unidade']
quantidade: {'description': 'Quantidade numérica, sem unidade', 'exclusiveMinimum': 0, 'title': 'Quantidade', 
'type': 'number'}


Configuração do cliente, igual ao 01 (célula de API: não roda sem `.env`):

In [ ]:
# api: configuração do cliente — lê a chave e o endpoint do .env
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("openai")
if not api_key:
    raise ValueError("variável 'openai' (chave da API) não encontrada no .env")
openai_base_url = os.getenv("openai_base_url")
if not openai_base_url:
    raise ValueError("variável 'openai_base_url' não encontrada no .env")

AI_MODEL = "gpt-4o-mini"
ai_client = OpenAI(api_key=api_key, base_url=openai_base_url)

## 2. Três jeitos de pedir a resposta no formato do contrato

### 2.1 Schema no prompt + validação local (funciona em qualquer endpoint)

In [4]:
def prompt_com_schema(modelo: type[BaseModel]) -> str:
    return (
        "Você gera listas de compras. Use ingredientes in natura sempre que possível e o sistema métrico.\n"
        "Responda SOMENTE com um JSON válido que siga este JSON Schema:\n"
        f"{json.dumps(modelo.model_json_schema(), ensure_ascii=False)}"
    )


mostrar_prompt("prompt_com_schema(ListaDeCompras)", prompt_com_schema(ListaDeCompras))

╭─ prompt_com_schema(ListaDeCompras) ────────────────────────────────────────────────────────────────────────╮
│ Você gera listas de compras. Use ingredientes in natura sempre que possível e o sistema métrico.           │
│ Responda SOMENTE com um JSON válido que siga este JSON Schema:                                             │
│ {                                                                                                          │
│   "$defs": {                                                                                               │
│     "Ingrediente": {                                                                                       │
│       "additionalProperties": false,                                                                       │
│       "properties": {                                                                                      │
│         "nome": {                                                                                          │
│

In [ ]:
# api: schema no prompt, validação com Pydantic
chat_completion = ai_client.chat.completions.create(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": prompt_com_schema(ListaDeCompras)},
        {"role": "user", "content": "Pizza margherita para 2 pessoas"},
    ],
)
resposta_ao_vivo = chat_completion.choices[0].message.content
lista_ao_vivo = ListaDeCompras.model_validate_json(resposta_ao_vivo)
mostrar(f"lista_ao_vivo: {lista_ao_vivo}")

### 2.2 Saída estruturada nativa (`.parse`) — se o endpoint suportar

Alguns provedores aceitam o modelo Pydantic direto e **restringem a geração** ao schema. O SDK valida e devolve o objeto pronto
em `.parsed`. Nem todo endpoint compatível com a OpenAI suporta isso; se o seu não suportar, use o 2.1.

In [ ]:
# api: saída estruturada nativa
completion = ai_client.chat.completions.parse(
    model=AI_MODEL,
    messages=[
        {"role": "system", "content": "Você gera listas de compras com ingredientes in natura e sistema métrico."},
        {"role": "user", "content": "Pizza margherita para 2 pessoas"},
    ],
    response_format=ListaDeCompras,
)
lista_parse = completion.choices[0].message.parsed
mostrar(f"lista_parse: {lista_parse}")

### 2.3 Com pydantic-ai (o `Agent` do seu notebook)

O `Agent` faz o mesmo com menos código: recebe `output_type=ListaDeCompras` e devolve o objeto validado. (O `OpenAIModel` do
exemplo original foi renomeado para `OpenAIChatModel`; o endpoint entra pelo `OpenAIProvider`.)

> **Pegadinha:** dentro do Jupyter use `await agente.run(...)`. O `agente.run_sync(...)` tenta abrir um event loop, e o kernel
> já tem um rodando: `RuntimeError: This event loop is already running`. Em script `.py` comum, `run_sync` funciona.

In [ ]:
# api: pydantic-ai com o mesmo endpoint
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

modelo_pai = OpenAIChatModel(AI_MODEL, provider=OpenAIProvider(base_url=openai_base_url, api_key=api_key))
agente_compras = Agent(modelo_pai, output_type=ListaDeCompras,
                       instructions="Você gera listas de compras com ingredientes in natura e sistema métrico.")
resultado = await agente_compras.run("Pizza margherita para 2 pessoas")  # no Jupyter: await, não run_sync
mostrar(f"resultado.output: {resultado.output}")

Sem chave, dá para ver o `Agent` funcionando com o `TestModel` do pydantic-ai, um modelo **falso** que gera dados que só
obedecem ao schema (os valores não fazem sentido — servem para testar o encanamento):

In [5]:
import os

os.environ["PYDANTIC_AI_NO_BANNER"] = "1"
from pydantic_ai import Agent
from pydantic_ai.models.test import TestModel

agente_teste = Agent(TestModel(), output_type=ListaDeCompras)
saida_teste = (await agente_teste.run("Pizza margherita para 2 pessoas")).output
mostrar(f"type(saida_teste).__name__: {type(saida_teste).__name__}")
mostrar(f"saida_teste: {saida_teste!r}")

type(saida_teste).__name__: ListaDeCompras
saida_teste: ListaDeCompras(prato='a', porcoes=1, ingredientes=[Ingrediente(nome='a', quantidade=1.0, 
unidade='a')])


## 3. Validando respostas

A resposta **real** gravada no 01 seguia o formato pedido lá: uma **lista** solta de `{"ingredients", "amount"}`, sem prato nem
porções. Contra o contrato novo, ela é recusada logo no primeiro nível:

In [6]:
resposta_antiga = Path("../respostas_gravadas/pizza_margherita_real.json").read_text(encoding="utf-8")
try:
    ListaDeCompras.model_validate_json(resposta_antiga)
except ValidationError as ex:
    mostrar(f"erros: {ex.error_count()}")
    mostrar(f"primeiro: {ex.errors()[0]['msg']}")

primeiro_item = json.loads(resposta_antiga)[0]
try:
    Ingrediente.model_validate(primeiro_item)
except ValidationError as ex:
    for erro in ex.errors():
        mostrar(f"{erro['loc']}: {erro['msg']}")

erros: 1
primeiro: Input should be an object
('nome',): Field required
('quantidade',): Field required
('unidade',): Field required
('ingredients',): Extra inputs are not permitted
('amount',): Extra inputs are not permitted


É por isso que o contrato **precisa ir para o modelo** (seções 2.1 a 2.3): o Pydantic confere, mas quem gera é o modelo.

Uma resposta no formato novo (*exemplo construído*, com as quantidades da resposta real):

In [7]:
resposta_nova = """
{"prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
  {"nome": "farinha de trigo", "quantidade": 250, "unidade": "g"},
  {"nome": "fermento biológico seco", "quantidade": 5, "unidade": "g"},
  {"nome": "água", "quantidade": 150, "unidade": "ml"},
  {"nome": "azeite", "quantidade": 15, "unidade": "ml"},
  {"nome": "molho de tomate", "quantidade": 100, "unidade": "g"},
  {"nome": "muçarela de búfala", "quantidade": 150, "unidade": "g"},
  {"nome": "manjericão fresco", "quantidade": 10, "unidade": "g"}
]}
"""
lista = ListaDeCompras.model_validate_json(resposta_nova)
mostrar(f"lista.prato: {lista.prato}")
mostrar(f"lista.porcoes: {lista.porcoes}")
mostrar(f"lista.ingredientes[0]: {lista.ingredientes[0]!r}")

lista.prato: Pizza margherita
lista.porcoes: 2
lista.ingredientes[0]: Ingrediente(nome='farinha de trigo', quantidade=250.0, unidade='g')


Agora o problema do 01 some: a quantidade é número.

In [8]:
def para_porcoes(lista: ListaDeCompras, porcoes: int) -> ListaDeCompras:
    fator = porcoes / lista.porcoes
    return ListaDeCompras(
        prato=lista.prato,
        porcoes=porcoes,
        ingredientes=[item.model_copy(update={"quantidade": item.quantidade * fator}) for item in lista.ingredientes],
    )


para_quatro = para_porcoes(lista, 4)
tabela(["ingrediente", "2 porções", "4 porções"],
       [(antes.nome, f"{antes.quantidade:g} {antes.unidade}", f"{depois.quantidade:g} {depois.unidade}")
        for antes, depois in zip(lista.ingredientes, para_quatro.ingredientes)])

                                                   
  ingrediente               2 porções   4 porções  
 ───────────────────────────────────────────────── 
  farinha de trigo          250 g       500 g      
  fermento biológico seco   5 g         10 g       
  água                      150 ml      300 ml     
  azeite                    15 ml       30 ml      
  molho de tomate           100 g       200 g      
  muçarela de búfala        150 g       300 g      
  manjericão fresco         10 g        20 g       
                                                   


## ⚠️ O que ainda quebra (e o [03](03_enums_e_guardrails.ipynb) resolve)

**1. "Unidade métrica" está só na descrição.** O campo `unidade` é `str`: qualquer texto passa (*exemplo construído*):

In [9]:
imperial = """
{"prato": "Pizza margherita", "porcoes": 2, "ingredientes": [
  {"nome": "flour", "quantidade": 2, "unidade": "cups"},
  {"nome": "olive oil", "quantidade": 1, "unidade": "tablespoon"}
]}
"""
aceita = ListaDeCompras.model_validate_json(imperial)
mostrar(f"unidades aceitas: {[item.unidade for item in aceita.ingredientes]}")

unidades aceitas: ['cups', 'tablespoon']


O `description` do campo é um **pedido** ao modelo, não uma regra. A validação passou e a lista está em xícaras.

**2. Texto em volta do JSON continua quebrando a seção 2.1.** O `model_validate_json` exige JSON puro:

In [10]:
com_cerca = 'Aqui está:\n```json\n' + resposta_nova.strip() + '\n```'
try:
    ListaDeCompras.model_validate_json(com_cerca)
except ValidationError as ex:
    mostrar(f"com cerca: {ex.errors()[0]['type']}")


def extrair_json(texto: str) -> str:
    """Pega o conteúdo entre a primeira '{' e a última '}' — simples, e suficiente para cercas e texto em volta."""
    inicio, fim = texto.find("{"), texto.rfind("}")
    if inicio == -1 or fim == -1:
        raise ValueError("nenhum objeto JSON na resposta")
    return texto[inicio:fim + 1]


mostrar(f"depois de extrair_json: {ListaDeCompras.model_validate_json(extrair_json(com_cerca)).prato}")

com cerca: json_invalid
depois de extrair_json: Pizza margherita


Na saída estruturada nativa (2.2) esse problema não existe; no caminho 2.1, um `extrair_json` simples resolve a maioria dos casos.

**Resumo:** o contrato garante a **forma** (campos e tipos). O vocabulário de cada campo — quais unidades, quais categorias — ainda
está solto. ➡️ [03 — enums e guardrails](03_enums_e_guardrails.ipynb)